In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm
import numpy as np
import pickle

class FFHallucinationClassifier(nn.Module):
    def __init__(self, input_shape, hidden_dim=256, dropout=0.2):
        super().__init__()

        self.linear_relu_stack = nn.Sequential(
            nn.Linear(input_shape, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits

In [2]:
def train_model(X_train, y_train, X_val, y_val, input_dim, hidden_dim=256, dropout=0.2,
                    learning_rate=1e-3, weight_decay=1e-4, batch_size=32, num_steps=1000, device="cpu"):
    model = FFHallucinationClassifier(
        input_shape=input_dim,
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)

    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)
    X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val, dtype=torch.long).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    model.train()

    for _ in range(num_steps):
        optimizer.zero_grad()

        sample = torch.randperm(X_train.shape[0], device=device)[:batch_size]

        pred = model(X_train[sample])
        loss = F.cross_entropy(pred, y_train[sample])

        loss.backward()
        optimizer.step()

    model.eval()

    with torch.no_grad():
        val_logits = model(X_val)
        val_probs = F.softmax(val_logits, dim=1)[:, 1]

        val_pred_classes = (val_probs > 0.5).long()

        val_auc = roc_auc_score(
            y_val.detach().cpu().numpy(),
            val_probs.detach().cpu().numpy()
        )

        val_acc = accuracy_score(y_val.detach().cpu().numpy(),
                                val_pred_classes.detach().cpu().numpy())

    return model, val_auc, val_acc

In [3]:
def make_splits(inputs, outputs):
    X_train, X_val, y_train, y_val = train_test_split(
        inputs,
        outputs,
        test_size=0.2,
        random_state=123,
        stratify=outputs
    )

    # X_train, X_val, y_train, y_val = train_test_split(
    #     X_temp,
    #     y_temp,
    #     test_size=0.2,
    #     random_state=123,
    #     stratify=y_temp
    # )

    return X_train, X_val, y_train, y_val

def random_search_tuning(
    inputs,
    outputs,
    n_trials=30,
    device="cpu",
    seed=123
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    X_train, X_val, y_train, y_val = make_splits(inputs, outputs)

    input_dim = X_train.shape[1]

    search_space = {
        "hidden_dim": [64, 128, 256, 512],
        "dropout": [0.0, 0.1, 0.2, 0.3, 0.5],
        "learning_rate": [1e-4, 3e-4, 1e-3, 3e-3],
        "weight_decay": [0.0, 1e-5, 1e-4, 1e-3],
        "batch_size": [16, 32, 64, 128],
        "num_steps": [500, 1000, 2000]
    }

    results = []

    best_auc = -1
    best_model = None
    best_params = None

    for trial in tqdm(range(n_trials)):
        params = {
            "hidden_dim": random.choice(search_space["hidden_dim"]),
            "dropout": random.choice(search_space["dropout"]),
            "learning_rate": random.choice(search_space["learning_rate"]),
            "weight_decay": random.choice(search_space["weight_decay"]),
            "batch_size": random.choice(search_space["batch_size"]),
            "num_steps": random.choice(search_space["num_steps"])
        }

        model, val_auc, val_acc = train_model(
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            input_dim=input_dim,
            hidden_dim=params["hidden_dim"],
            dropout=params["dropout"],
            learning_rate=params["learning_rate"],
            weight_decay=params["weight_decay"],
            batch_size=params["batch_size"],
            num_steps=params["num_steps"],
            device=device
        )

        result = {
            **params,
            "val_auc": val_auc,
            "val_acc": val_acc
        }

        results.append(result)

        if val_auc > best_auc:
            best_auc = val_auc
            best_model = model
            best_params = params

        print(f"Trial {trial + 1}/{n_trials}")
        print("Params:", params)
        print(f"Validation AUC: {val_auc:.4f}, Validation Acc: {val_acc:.4f}")
        print("-" * 60)

    results_df = pd.DataFrame(results).sort_values(
        by="val_auc",
        ascending=False
    )

    return best_model, best_params, results_df, (X_train, X_val, y_train, y_val)

In [4]:
pickle_file = "./results/1000_samples/open_llama_7b_trivia_qa_start-0_end-1000_4_28.pickle"
with open(pickle_file, "rb") as infile:
    results = pickle.loads(infile.read())

first_fc_features = np.stack([i[31] for i in results['first_fully_connected']])
correct = np.array(results['correct']).astype(int)
device = "cuda:1"

best_model, best_params, tuning_results, splits = random_search_tuning(
    inputs=first_fc_features,
    outputs=correct,
    n_trials=30,
    device=device
)

print("Best parameters:")
print(best_params)

print(tuning_results)

  3%|▎         | 1/30 [00:04<02:00,  4.15s/it]

Trial 1/30
Params: {'hidden_dim': 64, 'dropout': 0.2, 'learning_rate': 0.0001, 'weight_decay': 0.001, 'batch_size': 64, 'num_steps': 500}
Validation AUC: 0.6723, Validation Acc: 0.6450
------------------------------------------------------------


  7%|▋         | 2/30 [00:05<01:08,  2.46s/it]

Trial 2/30
Params: {'hidden_dim': 64, 'dropout': 0.3, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 16, 'num_steps': 500}
Validation AUC: 0.6528, Validation Acc: 0.6550
------------------------------------------------------------


 10%|█         | 3/30 [00:06<00:52,  1.95s/it]

...

Trial 28/30
Params: {'hidden_dim': 256, 'dropout': 0.5, 'learning_rate': 0.0003, 'weight_decay': 1e-05, 'batch_size': 64, 'num_steps': 1000}
Validation AUC: 0.6274, Validation Acc: 0.6450
------------------------------------------------------------


 97%|█████████▋| 29/30 [01:25<00:02,  2.61s/it]

Trial 29/30
Params: {'hidden_dim': 512, 'dropout': 0.0, 'learning_rate': 0.0001, 'weight_decay': 1e-05, 'batch_size': 32, 'num_steps': 500}
Validation AUC: 0.6355, Validation Acc: 0.6250
------------------------------------------------------------


100%|██████████| 30/30 [01:27<00:00,  2.91s/it]

Trial 30/30
Params: {'hidden_dim': 128, 'dropout': 0.2, 'learning_rate': 0.0003, 'weight_decay': 0.0, 'batch_size': 64, 'num_steps': 500}
Validation AUC: 0.6335, Validation Acc: 0.6400
------------------------------------------------------------
Best parameters:
{'hidden_dim': 64, 'dropout': 0.2, 'learning_rate': 0.0001, 'weight_decay': 0.001, 'batch_size': 64, 'num_steps': 500}
    hidden_dim  dropout  learning_rate  weight_decay  batch_size  num_steps  \
0           64      0.2         0.0001       0.00100          64        500   
4          512      0.0         0.0001       0.00000          32        500   
16          64      0.5         0.0010       0.00000          16       1000   
2          128      0.2         0.0010       0.00001          32        500   
11          64      0.2         0.0001       0.00010          32       1000   
26          64      0.5         0.0010       0.00001          64        500   
1           64      0.3         0.0010       0.00010          16 